<a href="https://colab.research.google.com/github/23064088/DataMiningGroup18/blob/naive-bayes/Naive_Bayes_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

#Load the Data
df = pd.read_csv('/content/sample_data/financial_data.csv')

In [2]:
df.head()

,Customer_ID,Age,Income,Credit_Score,Investment_Returns,Risk_Level,Customer_Feedback
0,1001,25.0,0.735552,0.127660,11.57,Medium,The app is found.
1,1002,29.0,0.530047,0.157447,7.75,Medium,The service was discontinued.
2,1003,25.0,0.293007,0.421277,11.80,High,Fees are high.
3,1004,41.0,1.000000,0.953191,9.46,Low,Fees are high.
4,1005,25.0,0.934614,0.221277,8.08,High,The app is found.


In [3]:
!pip install -q transformers accelerate bitsandbytes

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-v0.1"

# 1. The leanest possible configuration for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto", # Let the library decide the best path
    low_cpu_mem_usage=True
)

# 3. Create a very simple generator
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

def get_mistral_score(text):
    # Shortened prompt to save "token space" in memory
    prompt = f"Text: {text}\nSentiment (-1, 0, 1):"

    # max_new_tokens=2 makes it finish instantly
    res = generator(prompt, max_new_tokens=2, do_sample=False)
    out = res[0]['generated_text'].split("Sentiment (-1, 0, 1):")[-1].strip()

    if "-1" in out: return -1.0
    if "1" in out: return 1.0
    return 0.0

# 4. Process the data
print("Mistral is processing... this should fit now!")
df['Sentiment_Score'] = df['Customer_Feedback'].apply(get_mistral_score)

# Final check
df[['Customer_Feedback', 'Sentiment_Score']].head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Mistral is processing... this should fit now!


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting

,Customer_Feedback,Sentiment_Score
0,The app is found.,0.0
1,The service was discontinued.,0.0
2,Fees are high.,-1.0
3,Fees are high.,-1.0
4,The app is found.,0.0


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Prepare the data
# We use the Sentiment_Score Mistral just made!
X = df[['Age', 'Income', 'Credit_Score', 'Sentiment_Score']]
y = df['Risk_Level']

In [6]:
# 2. Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# 3. Train the Model
model = GaussianNB()
model.fit(X_train, y_train)

GaussianNB()

In [10]:
# 4. Get the results
y_pred = model.predict(X_test)
print(f"Final Model Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nDetailed Report:\n", classification_report(y_test, y_pred))

Final Model Accuracy: 42.00%

Detailed Report:
               precision    recall  f1-score   support

        High       0.36      0.33      0.35        60
         Low       0.44      0.51      0.48        86
      Medium       0.43      0.37      0.40        54

    accuracy                           0.42       200
   macro avg       0.41      0.41      0.41       200
weighted avg       0.42      0.42      0.42       200

